# 第1节:情感分析

本节介绍情感分析任务,包括数据集准备、基于RNN的模型和基于CNN的模型。

## 学习目标

1. 理解情感分析任务的定义和应用场景
2. 掌握IMDb电影评论数据集的处理流程
3. 实现基于双向LSTM的情感分类模型
4. 实现基于textCNN的情感分类模型
5. 比较RNN和CNN在文本分类中的优劣

## 1.1 情感分析简介

### 什么是情感分析?

随着在线社交媒体和评论平台的快速发展,大量评论的数据被记录下来。这些数据具有支持决策过程的巨大潜力。

**情感分析**(sentiment analysis)研究人们在文本中(如产品评论、博客评论和论坛讨论等)"隐藏"的情绪。它广泛应用于:

- **政治领域**: 公众对政策的情绪分析
- **金融领域**: 市场情绪分析
- **营销领域**: 产品研究和品牌管理

### 任务定义

由于情感可以被分类为离散的极性或尺度(例如,积极的和消极的),我们可以将**情感分析看作一项文本分类任务**,它将可变长度的文本序列转换为固定长度的文本类别。

### 数据集

我们使用斯坦福大学的**大型电影评论数据集**(large movie review dataset, IMDb)进行情感分析。

- 训练集和测试集各包含25,000个电影评论
- "积极"和"消极"标签数量相同,表示不同的情感极性
- 数据来源: IMDb电影评论网站

In [ ]:
import os
import re
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## 1.2 IMDb数据集准备

### 下载和读取数据集

首先,下载并提取IMDb评论数据集。

In [ ]:
#@save
d2l.DATA_HUB['aclImdb'] = (
    'http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz',
    '01ada507287d82875905620988597833ad4e0903')

data_dir = d2l.download_extract('aclImdb', 'aclImdb')

接下来,读取训练和测试数据集。每个样本都是一个评论及其标签:1表示"积极",0表示"消极"。

In [ ]:
#@save
def read_imdb(data_dir, is_train):
    """读取IMDb评论数据集文本序列和标签"""
    data, labels = [], []
    for label in ('pos', 'neg'):
        folder_name = os.path.join(data_dir, 'train' if is_train else 'test',
                                   label)
        for file in os.listdir(folder_name):
            with open(os.path.join(folder_name, file), 'rb') as f:
                review = f.read().decode('utf-8').replace('\n', '')
                data.append(review)
                labels.append(1 if label == 'pos' else 0)
    return data, labels

train_data = read_imdb(data_dir, is_train=True)
print('训练集数目:', len(train_data[0]))
for x, y in zip(train_data[0][:3], train_data[1][:3]):
    print('标签:', y, 'review:', x[0:60])

### 预处理数据集

将每个单词作为一个词元,过滤掉出现不到5次的单词,我们从训练数据集中创建一个词表。

In [ ]:
train_tokens = d2l.tokenize(train_data[0], token='word')
vocab = d2l.Vocab(train_tokens, min_freq=5, reserved_tokens=['<pad>'])

在词元化之后,让我们绘制评论词元长度的直方图。

In [ ]:
d2l.set_figsize()
d2l.plt.xlabel('# tokens per review')
d2l.plt.ylabel('count')
d2l.plt.hist([len(line) for line in train_tokens], bins=range(0, 1000, 50));

正如我们所料,评论的长度各不相同。为了每次处理一小批量这样的评论,我们通过截断和填充将每个评论的长度设置为500。

In [ ]:
num_steps = 500  # 序列长度
train_features = torch.tensor([d2l.truncate_pad(
    vocab[line], num_steps, vocab['<pad>']) for line in train_tokens])
print(train_features.shape)

### 创建数据迭代器

现在我们可以创建数据迭代器了。在每次迭代中,都会返回一小批量样本。

In [ ]:
train_iter = d2l.load_array((train_features,
    torch.tensor(train_data[1])), 64)

for X, y in train_iter:
    print('X:', X.shape, ', y:', y.shape)
    break
print('小批量数目:', len(train_iter))

### 整合代码

最后,我们将上述步骤封装到`load_data_imdb`函数中。它返回训练和测试数据迭代器以及IMDb评论数据集的词表。

In [ ]:
#@save
def load_data_imdb(batch_size, num_steps=500):
    """返回数据迭代器和IMDb评论数据集的词表"""
    data_dir = d2l.download_extract('aclImdb', 'aclImdb')
    train_data = read_imdb(data_dir, True)
    test_data = read_imdb(data_dir, False)
    train_tokens = d2l.tokenize(train_data[0], token='word')
    test_tokens = d2l.tokenize(test_data[0], token='word')
    vocab = d2l.Vocab(train_tokens, min_freq=5)
    train_features = torch.tensor([d2l.truncate_pad(
        vocab[line], num_steps, vocab['<pad>']) for line in train_tokens])
    test_features = torch.tensor([d2l.truncate_pad(
        vocab[line], num_steps, vocab['<pad>']) for line in test_tokens])
    train_iter = d2l.load_array((train_features, torch.tensor(train_data[1])),
                                batch_size)
    test_iter = d2l.load_array((test_features, torch.tensor(test_data[1])),
                               batch_size,
                               is_train=False)
    return train_iter, test_iter, vocab

## 1.3 使用循环神经网络的情感分析

### 模型架构

与词相似度和类比任务一样,我们也可以将预先训练的词向量应用于情感分析。由于IMDb评论数据集不是很大,使用在大规模语料库上预训练的文本表示可以减少模型的过拟合。

![将GloVe送入基于循环神经网络的架构,用于情感分析](https://zh.d2l.ai/_images/nlp-map-sa-rnn.svg)

我们将使用预训练的GloVe模型来表示每个词元,并将这些词元表示送入**多层双向循环神经网络**以获得文本序列表示,该文本序列表示将被转换为情感分析输出。

In [ ]:
batch_size = 64
train_iter, test_iter, vocab = d2l.load_data_imdb(batch_size)

### 使用循环神经网络表示单个文本

在文本分类任务(如情感分析)中,可变长度的文本序列将被转换为固定长度的类别。在下面的`BiRNN`类中:

- 文本序列的每个词元经由**嵌入层**(`self.embedding`)获得其单独的预训练GloVe表示
- 整个序列由**双向循环神经网络**(`self.encoder`)编码
- 双向LSTM在初始和最终时间步的隐状态被连结起来作为文本序列的表示
- 通过一个具有两个输出("积极"和"消极")的**全连接层**(`self.decoder`),将此单一文本表示转换为输出类别

In [ ]:
class BiRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, num_hiddens,
                 num_layers, **kwargs):
        super(BiRNN, self).__init__(**kwargs)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        # 将bidirectional设置为True以获取双向循环神经网络
        self.encoder = nn.LSTM(embed_size, num_hiddens, num_layers=num_layers,
                                bidirectional=True)
        self.decoder = nn.Linear(4 * num_hiddens, 2)

    def forward(self, inputs):
        # inputs的形状是(批量大小,时间步数)
        # 因为长短期记忆网络要求其输入的第一个维度是时间维,
        # 所以在获得词元表示之前,输入会被转置。
        # 输出形状为(时间步数,批量大小,词向量维度)
        embeddings = self.embedding(inputs.T)
        self.encoder.flatten_parameters()
        # 返回上一个隐藏层在不同时间步的隐状态,
        # outputs的形状是(时间步数,批量大小,2*隐藏单元数)
        outputs, _ = self.encoder(embeddings)
        # 连结初始和最终时间步的隐状态,作为全连接层的输入,
        # 其形状为(批量大小,4*隐藏单元数)
        encoding = torch.cat((outputs[0], outputs[-1]), dim=1)
        outs = self.decoder(encoding)
        return outs

让我们构造一个具有两个隐藏层的双向循环神经网络来表示单个文本以进行情感分析。

In [ ]:
embed_size, num_hiddens, num_layers = 100, 100, 2
devices = d2l.try_all_gpus()
net = BiRNN(len(vocab), embed_size, num_hiddens, num_layers)

In [ ]:
def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
    if type(m) == nn.LSTM:
        for param in m._flat_weights_names:
            if "weight" in param:
                nn.init.xavier_uniform_(m._parameters[param])
net.apply(init_weights);

### 加载预训练的词向量

下面,我们为词表中的单词加载预训练的100维(需要与`embed_size`一致)的GloVe嵌入。

In [ ]:
glove_embedding = d2l.TokenEmbedding('glove.6b.100d')

打印词表中所有词元向量的形状。

In [ ]:
embeds = glove_embedding[vocab.idx_to_token]
embeds.shape

我们使用这些预训练的词向量来表示评论中的词元,并且在训练期间不要更新这些向量。

In [ ]:
net.embedding.weight.data.copy_(embeds)
net.embedding.weight.requires_grad = False

### 训练和评估模型

现在我们可以训练双向循环神经网络进行情感分析。

In [ ]:
lr, num_epochs = 0.01, 5
trainer = torch.optim.Adam(net.parameters(), lr=lr)
loss = nn.CrossEntropyLoss(reduction="none")
d2l.train_ch13(net, train_iter, test_iter, loss, trainer, num_epochs,
    devices)

我们定义以下函数来使用训练好的模型`net`预测文本序列的情感。

In [ ]:
#@save
def predict_sentiment(net, vocab, sequence):
    """预测文本序列的情感"""
    sequence = torch.tensor(vocab[sequence.split()], device=d2l.try_gpu())
    label = torch.argmax(net(sequence.reshape(1, -1)), dim=1)
    return 'positive' if label == 1 else 'negative'

最后,让我们使用训练好的模型对两个简单的句子进行情感预测。

In [ ]:
predict_sentiment(net, vocab, 'this movie is so great')

In [ ]:
predict_sentiment(net, vocab, 'this movie is so bad')

## 1.4 使用卷积神经网络的情感分析

### textCNN模型简介

虽然卷积神经网络最初是为计算机视觉设计的,但它也被广泛用于自然语言处理。简单地说,只要**将任何文本序列想象成一维图像**即可。通过这种方式,一维卷积神经网络可以处理文本中的局部特征,例如$n$元语法。

![将GloVe放入卷积神经网络架构进行情感分析](https://zh.d2l.ai/_images/nlp-map-sa-cnn.svg)

与使用带有GloVe预训练的循环神经网络架构进行情感分析相比,唯一的区别在于架构的选择。

### 一维卷积

在介绍该模型之前,让我们先看看一维卷积是如何工作的。请记住,这只是基于互相关运算的二维卷积的特例。

![一维互相关运算](https://zh.d2l.ai/_images/conv1d.svg)

在一维情况下,卷积窗口在输入张量上从左向右滑动。在滑动期间,卷积窗口中某个位置包含的输入子张量和核张量按元素相乘。这些乘法的总和在输出张量的相应位置给出单个标量值。

In [ ]:
def corr1d(X, K):
    w = K.shape[0]
    Y = torch.zeros((X.shape[0] - w + 1))
    for i in range(Y.shape[0]):
        Y[i] = (X[i: i + w] * K).sum()
    return Y

我们可以验证一维互相关实现的输出。

In [ ]:
X, K = torch.tensor([0, 1, 2, 3, 4, 5, 6]), torch.tensor([1, 2])
corr1d(X, K)

对于任何具有多个通道的一维输入,卷积核需要具有相同数量的输入通道。然后,对于每个通道,对输入的一维张量和卷积核的一维张量执行互相关运算,将所有通道上的结果相加以产生一维输出张量。

![具有3个输入通道的一维互相关运算](https://zh.d2l.ai/_images/conv1d-channel.svg)

In [ ]:
def corr1d_multi_in(X, K):
    # 首先,遍历'X'和'K'的第0维(通道维)。然后,把它们加在一起
    return sum(corr1d(x, k) for x, k in zip(X, K))

X = torch.tensor([[0, 1, 2, 3, 4, 5, 6],
              [1, 2, 3, 4, 5, 6, 7],
              [2, 3, 4, 5, 6, 7, 8]])
K = torch.tensor([[1, 2], [3, 4], [-1, -3]])
corr1d_multi_in(X, K)

### textCNN模型架构

使用一维卷积和最大时间汇聚,textCNN模型将单个预训练的词元表示作为输入,然后获得并转换用于下游应用的序列表示。

对于具有由$d$维向量表示的$n$个词元的单个文本序列,输入张量的宽度、高度和通道数分别为$n$、$1$和$d$。textCNN模型将输入转换为输出,如下所示:

1. 定义多个一维卷积核,并分别对输入执行卷积运算。具有不同宽度的卷积核可以捕获不同数目的相邻词元之间的局部特征。
2. 在所有输出通道上执行最大时间汇聚层,然后将所有标量汇聚输出连结为向量。
3. 使用全连接层将连结后的向量转换为输出类别。Dropout可以用来减少过拟合。

![textCNN的模型架构](https://zh.d2l.ai/_images/textcnn.svg)

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_size, kernel_sizes, num_channels,
                 **kwargs):
        super(TextCNN, self).__init__(**kwargs)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        # 这个嵌入层不需要训练
        self.constant_embedding = nn.Embedding(vocab_size, embed_size)
        self.dropout = nn.Dropout(0.5)
        self.decoder = nn.Linear(sum(num_channels), 2)
        # 最大时间汇聚层没有参数,因此可以共享此实例
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.relu = nn.ReLU()
        # 创建多个一维卷积层
        self.convs = nn.ModuleList()
        for c, k in zip(num_channels, kernel_sizes):
            self.convs.append(nn.Conv1d(2 * embed_size, c, k))

    def forward(self, inputs):
        # 沿着向量维度将两个嵌入层连结起来,
        # 每个嵌入层的输出形状都是(批量大小,词元数量,词元向量维度)连结起来
        embeddings = torch.cat((
            self.embedding(inputs), self.constant_embedding(inputs)), dim=2)
        # 根据一维卷积层的输入格式,重新排列张量,以便通道作为第2维
        embeddings = embeddings.permute(0, 2, 1)
        # 每个一维卷积层在最大时间汇聚层合并后,获得的张量形状是(批量大小,通道数,1)
        # 删除最后一个维度并沿通道维度连结
        encoding = torch.cat([
            torch.squeeze(self.relu(self.pool(conv(embeddings))), dim=-1)
            for conv in self.convs], dim=1)
        outputs = self.decoder(self.dropout(encoding))
        return outputs

让我们创建一个textCNN实例。它有3个卷积层,卷积核宽度分别为3、4和5,均有100个输出通道。

In [ ]:
embed_size, kernel_sizes, nums_channels = 100, [3, 4, 5], [100, 100, 100]
devices = d2l.try_all_gpus()
net = TextCNN(len(vocab), embed_size, kernel_sizes, nums_channels)

def init_weights(m):
    if type(m) in (nn.Linear, nn.Conv1d):
        nn.init.xavier_uniform_(m.weight)

net.apply(init_weights);

### 加载预训练词向量

我们加载预训练的100维GloVe嵌入作为初始化的词元表示。这些词元表示(嵌入权重)在`embedding`中将被训练,在`constant_embedding`中将被固定。

In [ ]:
glove_embedding = d2l.TokenEmbedding('glove.6b.100d')
embeds = glove_embedding[vocab.idx_to_token]
net.embedding.weight.data.copy_(embeds)
net.constant_embedding.weight.data.copy_(embeds)
net.constant_embedding.weight.requires_grad = False

### 训练和评估模型

现在我们可以训练textCNN模型进行情感分析。

In [ ]:
lr, num_epochs = 0.001, 5
trainer = torch.optim.Adam(net.parameters(), lr=lr)
loss = nn.CrossEntropyLoss(reduction="none")
d2l.train_ch13(net, train_iter, test_iter, loss, trainer, num_epochs, devices)

下面,我们使用训练好的模型来预测两个简单句子的情感。

In [ ]:
d2l.predict_sentiment(net, vocab, 'this movie is so great')

In [ ]:
d2l.predict_sentiment(net, vocab, 'this movie is so bad')

## 1.5 RNN vs CNN对比

### 模型对比

| 方面 | RNN模型(BiLSTM) | CNN模型(textCNN) |
|------|----------------|------------------|
| **核心思想** | 序列建模,捕获上下文依赖 | 局部特征提取,捕获n-gram模式 |
| **优点** | 1. 能够捕获长距离依赖<br>2. 保留序列顺序信息<br>3. 理论上可以处理任意长度序列 | 1. 并行计算效率高<br>2. 能够捕获多尺度局部特征<br>3. 训练速度快 |
| **缺点** | 1. 训练速度慢(序列处理)<br>2. 容易出现梯度消失/爆炸<br>3. 需要更多的训练时间 | 1. 难以捕获长距离依赖<br>2. 对词序信息的利用不如RNN<br>3. 卷积核大小需要预先设定 |
| **适用场景** | 1. 需要捕获长距离依赖的任务<br>2. 序列标注任务<br>3. 对序列顺序敏感的任务 | 1. 文本分类任务<br>2. 需要快速推理的场景<br>3. 局部特征重要的任务 |
| **计算复杂度** | O(n) - 序列长度 | O(1) - 可并行 |
| **参数量** | 较多(LSTM单元) | 较少(卷积核) |

### 实验对比

在IMDb情感分析任务上:
- **BiRNN**: 使用双向LSTM,捕获前后文信息,连结首尾时间步隐状态
- **textCNN**: 使用多个不同尺寸的卷积核(3,4,5),提取不同n-gram特征

两种方法都使用了:
- 预训练的GloVe词向量(100维)
- Dropout正则化
- 相似的训练策略

### 选择建议

1. **优先使用CNN**,如果:
   - 主要关注局部特征(如关键词、短语)
   - 需要快速训练和推理
   - 文本长度适中

2. **优先使用RNN**,如果:
   - 需要理解长距离依赖关系
   - 序列顺序信息至关重要
   - 可以接受较长的训练时间

3. **结合使用**:
   - 可以设计混合模型,结合两者优势
   - 例如:CNN提取局部特征,RNN建模序列依赖

## 小结

### 情感分析任务

- 情感分析研究人们在文本中的情感,这被认为是一个**文本分类问题**
- 将可变长度的文本序列转换为固定长度的文本类别
- 广泛应用于政治、金融、营销等领域

### 数据处理

- 经过预处理后,我们可以使用词表将IMDb评论数据集加载到数据迭代器中
- 需要进行词元化、截断/填充、创建词表等步骤

### RNN方法

- 预训练的词向量可以表示文本序列中的各个词元
- 双向循环神经网络可以表示文本序列
- 通过连结初始和最终时间步的隐状态,可以使用全连接的层将该单个文本表示转换为类别

### CNN方法

- 一维卷积神经网络可以处理文本中的局部特征,例如$n$元语法
- 多输入通道的一维互相关等价于单输入通道的二维互相关
- 最大时间汇聚层允许在不同通道上使用不同数量的时间步长
- textCNN模型使用一维卷积层和最大时间汇聚层将单个词元表示转换为下游应用输出

### 实践建议

1. 使用预训练词向量可以减少过拟合
2. 根据任务特点选择合适的模型架构
3. 注意调整超参数(学习率、批量大小、序列长度等)
4. 可以尝试ensemble方法结合不同模型的预测

## 练习

1. 我们可以修改哪些超参数来加速训练情感分析模型?

2. 请实现一个函数来将Amazon reviews的数据集加载到数据迭代器中进行情感分析。

3. 增加迭代轮数可以提高训练和测试的准确性吗?调优其他超参数怎么样?

4. 使用较大的预训练词向量,例如300维的GloVe嵌入。它是否提高了分类精度?

5. 是否可以通过spaCy词元化来提高分类精度?

6. 调整超参数,并比较RNN和CNN架构在分类精度和计算效率方面的差异。

7. 在输入表示中添加位置编码。它是否提高了分类的精度?